# MVA Track 2 · NB10 v1.0.1
## Rescue-rule operating characteristics with a replicate-level statistical gate

This notebook is a **synthetic methodology audit**, not evidence of drug efficacy.

A clean rescue call requires all of:
1. ≥25% relative reduction in chromosome missegregation per completed division;
2. one-sided Welch test on biological-replicate rates, `p ≤ 0.05`;
3. no >10% reduction in completed-division output;
4. no >10 percentage-point increase in apoptosis;
5. ≥100 completed divisions per replicate.

The 25% reduction threshold is a simulation operating point, not a validated biological or clinical cutoff.


### Why this correction was needed

NB10 v1.0.0 used effect size alone and produced an unacceptably high null false-rescue rate.
v1.0.1 adds an inferential gate while keeping biological replicate, not individual mitosis, as the unit of statistical evidence.


In [ ]:
import numpy as np, pandas as pd
from scipy.stats import t as tdist
RNG_SEED=20260925
N_SIM=5000
ICC=0.02
RRR_GATE=0.25
ALPHA=0.05
MIN_DIV=100


In [ ]:
def beta_params(mean,icc):
    total=1/icc-1
    return mean*total,(1-mean)*total

def simulate_matrix(rng,mean_p,n,reps,nsim=N_SIM,icc=ICC):
    a,b=beta_params(mean_p,icc)
    ps=rng.beta(a,b,size=(nsim,reps))
    return rng.binomial(n,ps)/n

def welch_one_sided(control,treated):
    m0=control.mean(axis=1); m1=treated.mean(axis=1)
    v0=control.var(axis=1,ddof=1); v1=treated.var(axis=1,ddof=1)
    n0=control.shape[1]; n1=treated.shape[1]
    se2=v0/n0+v1/n1
    denom=(v0/n0)**2/(n0-1)+(v1/n1)**2/(n1-1)
    tstat=np.divide(m0-m1,np.sqrt(se2),out=np.zeros_like(m0),where=se2>0)
    df=np.divide(se2**2,denom,out=np.full_like(m0,n0+n1-2,dtype=float),where=denom>0)
    return tdist.sf(tstat,df)


In [ ]:
def sim(p0,p1,n,reps,out=1.0,apop=0.0,seed=1):
    rng=np.random.default_rng(seed)
    c=simulate_matrix(rng,p0,n,reps)
    nt=max(1,int(round(n*out)))
    t=simulate_matrix(rng,p1,nt,reps)
    cm=c.mean(1); tm=t.mean(1)
    rrr=np.divide(cm-tm,cm,out=np.zeros_like(cm),where=cm>0)
    p=welch_one_sided(c,t)
    call=(rrr>=RRR_GATE)&(p<=ALPHA)&(out>=.90)&(apop<=10.0)&(nt>=MIN_DIV)
    return dict(clean_rescue_call_rate=float(call.mean()),
                median_rrr=float(np.median(rrr)),
                median_welch_p=float(np.median(p)),
                cytostasis_flag=bool(out<.90),
                selective_death_flag=bool(apop>10.0))


In [ ]:
scenarios=[
("NO_RESCUE",.20,.20,1.0,0.0),
("TRUE_RESCUE_SMALL",.20,.16,1.0,0.0),
("TRUE_RESCUE_MODERATE",.20,.12,1.0,0.0),
("TRUE_RESCUE_STRONG",.20,.10,1.0,0.0),
("CYTOSTASIS_ONLY",.20,.20,.75,0.0),
("APPARENT_RESCUE_WITH_CYTOSTASIS",.20,.12,.80,0.0),
("APPARENT_RESCUE_WITH_SELECTIVE_KILLING",.20,.12,1.0,20.0),
]
rows=[]
for reps in [4,6,8]:
    for n in [100,150,250]:
        for i,(name,p0,p1,out,apop) in enumerate(scenarios):
            z=sim(p0,p1,n,reps,out,apop,seed=RNG_SEED+reps*10000+n+i)
            rows.append(dict(scenario=name,biological_replicates=reps,
                             completed_divisions_per_rep=n,baseline_rate=p0,
                             treated_rate=p1,true_rrr=(p0-p1)/p0,
                             output_ratio=out,apoptosis_delta_pp=apop,
                             rrr_gate=RRR_GATE,alpha=ALPHA,beta_binomial_icc=ICC,**z))
RESULTS=pd.DataFrame(rows)
display(RESULTS)


In [ ]:
assert (RESULTS[RESULTS.scenario=="NO_RESCUE"].clean_rescue_call_rate<=0.05).all()
assert (RESULTS[RESULTS.scenario.isin([
    "CYTOSTASIS_ONLY","APPARENT_RESCUE_WITH_CYTOSTASIS",
    "APPARENT_RESCUE_WITH_SELECTIVE_KILLING"
])].clean_rescue_call_rate==0).all()
print("NB10 v1.0.1 statistical and false-rescue gates: PASS")


## Interpretation contract

NB10 may be used to discuss assay operating characteristics and false-rescue protection.

It may not be cited as evidence that arimoclomol, 4-PBA, GGA, ambroxol, or any other drug rescues BUB1B/MVA.
The simulation also shows that four biological replicates are a minimum QC floor rather than a high-power design for moderate effects under replicate heterogeneity.
